In [ ]:
# delete_old_mlflow.py
from mlflow.tracking import MlflowClient
from urllib.parse import urlparse
import shutil
import os
import time
import argparse


def ms_from_days(days):
    return int(days * 24 * 3600 * 1000)


def delete_local_artifacts_if_exists(artifact_uri):
    # only handle file:// URIs (local paths)
    if artifact_uri is None:
        return
    p = artifact_uri
    if p.startswith("file://"):
        p = p[len("file://"):]
    # safety: require that path contains "mlruns" or another expected substring
    if "mlruns" not in p and "/artifacts" not in p:
        print("WARN: artifact path looks unexpected, skipping delete:", p)
        return
    if os.path.exists(p):
        print("Removing local artifact dir:", p)
        shutil.rmtree(p)
    else:
        print("Artifact path not found (skipping):", p)


def main(tracking_uri=None, cutoff_days=365, dry_run=True, remove_artifacts=False):
    client = MlflowClient(tracking_uri) if tracking_uri else MlflowClient()
    experiments = client.list_experiments()  # returns Experiment objects
    cutoff_ms = int(time.time() * 1000) - ms_from_days(cutoff_days)
    print("Cutoff (ms):", cutoff_ms)

    for exp in experiments:
        exp_id = exp.experiment_id
        print(f"EXP {exp_id} : {exp.name} (lifecycle={exp.lifecycle_stage})")
        # list runs for experiment
        runs = client.search_runs(
            [exp_id], filter_string="", max_results=100000)
        to_delete = []
        for r in runs:
            run_id = r.info.run_id
            start_time = getattr(r.info, "start_time", None)  # ms
            if start_time is None:
                # fallback: try to parse from tags/metrics if you have them
                continue
            if start_time < cutoff_ms:
                to_delete.append((run_id, r.info.artifact_uri))
        print(
            f"  Found {len(to_delete)} runs to delete in experiment {exp_id}")

        for run_id, artifact_uri in to_delete:
            print("  Deleting run:", run_id)
            if dry_run:
                print(
                    "    (dry-run) would delete run and optionally artifacts:", artifact_uri)
            else:
                client.delete_run(run_id)  # soft delete
                if remove_artifacts:
                    delete_local_artifacts_if_exists(artifact_uri)

        # Optional: if you want to delete whole experiment (must be empty logically)
        # if not dry_run:
        #     client.delete_experiment(exp_id)


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--tracking-uri", default=None,
                        help="mlflow tracking uri (optional)")
    parser.add_argument("--cutoff-days", type=int, default=365,
                        help="delete runs older than N days")
    parser.add_argument("--dry-run", action="store_true",
                        default=True, help="do not actually delete runs")
    parser.add_argument("--remove-artifacts", action="store_true", default=False,
                        help="try to remove local artifact dirs (only file://)")
    args = parser.parse_args()
    main(args.tracking_uri, args.cutoff_days, dry_run=args.dry_run,
         remove_artifacts=args.remove_artifacts)

In [1]:
import glob
import os
import sys

try:
    sys.path.append(
        '/home/28s_mur@lab.graphicon.ru/carla/PythonAPI/carla/dist/carla-0.9.9-py3.7-linux-x86_64.egg')
except IndexError:
    print('kys')

import carla
import random
import cv2
import skimage.measure as measure

# in synchronous mode, sensor data must be added to a queue
import queue
client = carla.Client('localhost', 2000)
client.set_timeout(11.0)
world = client.load_world('Town03')
settings = world.get_settings()
# must be less than 0.1, or else physics will be noisy
settings.fixed_delta_seconds = 0.05
# must use fixed delta seconds and synchronous mode for python api controlled sim, or else
# camera and sensor data may not match simulation properly and will be noisy
settings.synchronous_mode = True
world.apply_settings(settings)

: 

In [4]:
import os
import yaml
from datetime import datetime, timedelta
import shutil

# --- CONFIG ---
MLRUNS_PATH = "mlruns/914879025403898454"  # path to the experiment folder
DAYS_THRESHOLD = 7                           # delete runs newer than this
DELETE = True                               # Set False for dry run

# --- Calculate cutoff timestamp ---
cutoff_time = datetime.now() - timedelta(days=DAYS_THRESHOLD)
cutoff_timestamp = int(cutoff_time.timestamp() * 1000)  # MLflow uses ms in meta.yaml

# --- Collect runs ---
runs = []
for run_id in os.listdir(MLRUNS_PATH):
    run_path = os.path.join(MLRUNS_PATH, run_id)
    meta_file = os.path.join(run_path, "meta.yaml")
    
    if not os.path.isfile(meta_file):
        continue
    
    try:
        with open(meta_file, "r") as f:
            meta = yaml.safe_load(f)
        
        start_time = meta.get("start_time")
        run_name = meta.get("run_name", "N/A")
        if start_time is None:
            print(f"Skipping {run_id}: no start_time found")
            continue
        
        runs.append((start_time, run_name, run_id, run_path))
    except Exception as e:
        print(f"Error reading {run_id}/meta.yaml: {e}")

# --- Sort runs by start_time ascending ---
runs.sort(key=lambda x: x[0])

# --- Iterate and delete ---
for start_time, run_name, run_id, run_path in runs:
    if start_time >= cutoff_timestamp:
        if DELETE:
            print(f"Deleting run {run_name}, {run_id}, started at {datetime.fromtimestamp(start_time/1000)}")
            shutil.rmtree(run_path)
        else:
            print(f"[DRY RUN] Would delete run {run_name}, {run_id}, started at {datetime.fromtimestamp(start_time/1000)}")


Deleting run train_raft_patch_Kitti15_2025-12-10_17-46-45, 47982da4549548afb8ce41104bcb42b6, started at 2025-12-10 17:46:45.724000
Deleting run train_raft_patch_Kitti15_2025-12-10_17-48-23, 9495a47a954a4dbd922e326d649abe6f, started at 2025-12-10 17:48:23.526000
Deleting run train_raft_patch_Kitti15_2025-12-10_17-52-23, 04221640358f40f584067505d456e958, started at 2025-12-10 17:52:23.438000
Deleting run train_raft_patch_Kitti15_2025-12-10_17-55-42, 04a8265dd33b4b498a547752f2b01123, started at 2025-12-10 17:55:42.431000
Deleting run train_raft_patch_Kitti15_2025-12-10_17-59-59, 7493886a6b534ad28b9ab5f1b12bc9c9, started at 2025-12-10 17:59:59.487000
Deleting run train_raft_patch_Kitti15_2025-12-10_18-05-53, b32c2c8f6054455188681e28d4b292aa, started at 2025-12-10 18:05:53.771000
Deleting run train_raft_patch_Kitti15_2025-12-10_18-38-08, 7ddf5ea1bb6c4e1f89699fdd7c9aa65a, started at 2025-12-10 18:38:08.997000
Deleting run train_raft_patch_Kitti15_2025-12-10_18-39-12, 7298f049ddcc423bb4865892